In [1]:
import aimet_onnx
print(aimet_onnx.__version__)

2.29.0


In [2]:
import os
import onnx
import torch
from torchvision.models import mobilenet_v2

model = mobilenet_v2(weights='DEFAULT').eval()

dummy_input = torch.randn((10, 3, 224, 224))
file_path = os.path.join('/tmp', f'mobilenet_v2.onnx')
torch.onnx.export(model, dummy_input, file_path, dynamo=False)
onnx_model = onnx.load_model(file_path)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /home/harish/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|███████████████████████████████████████████████████████████████████████████| 13.6M/13.6M [00:05<00:00, 2.77MB/s]
/tmp/ipykernel_853/223029029.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(model, dummy_input, file_path, dynamo=False)


In [3]:
from aimet_onnx.quantsim import QuantizationSimModel
from aimet_onnx import int8, int16
from aimet_onnx.utils import make_dummy_input

sim = QuantizationSimModel(onnx_model,
                           param_type=int8,
                           activation_type=int16)

2026-04-29 21:49:38,288 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:None


In [4]:
calibration_data = make_dummy_input(onnx_model)
sim.compute_encodings(inputs=[calibration_data])

In [5]:
input_name = tuple(calibration_data.keys())[0]
output = sim.session.run(None, { input_name : dummy_input.numpy() })
print(output)

[array([[-0.34569559,  0.25262675,  0.44460097, ..., -0.40193126,
        -0.1142854 ,  0.13360901],
       [-0.41936195,  0.15987337,  0.609601  , ..., -0.7416326 ,
        -0.06885518,  0.25483516],
       [-0.319589  ,  0.28094175,  0.40169466, ..., -0.49160862,
        -0.01214627,  0.22565255],
       ...,
       [-0.32511002,  0.3102821 ,  0.57103264, ..., -0.37976825,
        -0.12966542, -0.10308558],
       [-0.4344265 ,  0.19197424,  0.5259968 , ..., -0.5077774 ,
        -0.0496893 ,  0.06222994],
       [-0.43221807,  0.24789442,  0.66812396, ..., -0.60636723,
        -0.03817401,  0.14465109]], dtype=float32)]


In [6]:
import os
import time
import numpy as np
import torch
import onnx
import onnxruntime as ort

from torchvision.models import mobilenet_v2

from aimet_onnx.quantsim import QuantizationSimModel
from aimet_onnx import int8
from aimet_onnx.utils import make_dummy_input

# -----------------------------
# 1. Export FP32 model
# -----------------------------
model = mobilenet_v2(weights='DEFAULT').eval()

dummy_input = torch.randn(1, 3, 224, 224)

onnx_path = "mobilenet_fp32.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print(f"[INFO] FP32 ONNX saved: {onnx_path}")

# -----------------------------
# 2. FP32 Inference (CPU)
# -----------------------------
session_fp32 = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])

input_name = session_fp32.get_inputs()[0].name

input_data = dummy_input.numpy()

start = time.time()
fp32_output = session_fp32.run(None, {input_name: input_data})
fp32_time = time.time() - start

print(f"[FP32] Inference Time: {fp32_time:.6f} sec")

# -----------------------------
# 3. AIMET QuantSim (INT8)
# -----------------------------
onnx_model = onnx.load_model(onnx_path)

sim = QuantizationSimModel(
    onnx_model,
    param_type=int8,
    activation_type=int8
)

# -----------------------------
# 4. Calibration
# -----------------------------
calib_data = make_dummy_input(onnx_model)

sim.compute_encodings(inputs=[calib_data])

print("[INFO] Calibration done")

# -----------------------------
# 5. INT8 Inference (Simulated)
# -----------------------------
sim_session = sim.session
input_name_sim = list(calib_data.keys())[0]

start = time.time()
int8_output = sim_session.run(None, {input_name_sim: input_data})
int8_time = time.time() - start

print(f"[INT8 Sim] Inference Time: {int8_time:.6f} sec")

# -----------------------------
# 6. Accuracy Comparison
# -----------------------------
fp32_out = fp32_output[0]
int8_out = int8_output[0]

mse = np.mean((fp32_out - int8_out) ** 2)
max_diff = np.max(np.abs(fp32_out - int8_out))

print(f"[COMPARE] MSE: {mse}")
print(f"[COMPARE] Max Abs Diff: {max_diff}")

# -----------------------------
# 7. Save Quantized Model
# -----------------------------
export_path = "./quantized_model"
os.makedirs(export_path, exist_ok=True)

sim.export(
    path=export_path,
    filename_prefix="mobilenet_int8"
)

print(f"[INFO] Quantized model saved at: {export_path}")

# Files generated:
# mobilenet_int8.onnx
# mobilenet_int8.encodings

# -----------------------------
# 8. Model Size Comparison
# -----------------------------
fp32_size = os.path.getsize(onnx_path) / (1024 * 1024)
int8_size = os.path.getsize(os.path.join(export_path, "mobilenet_int8.onnx")) / (1024 * 1024)

print(f"[SIZE] FP32 Model Size: {fp32_size:.2f} MB")
print(f"[SIZE] INT8 Model Size (sim): {int8_size:.2f} MB")

W0429 21:58:05.269000 853 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/harish/miniconda3/envs/qairt/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Assertion `node->hasAttribute(kaxes)` failed: No initializer or constant input to node found


[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
[INFO] FP32 ONNX saved: mobilenet_fp32.onnx
[FP32] Inference Time: 0.009537 sec
2026-04-29 21:58:12,378 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:None
[INFO] Calibration done
[INT8 Sim] Inference Time: 0.030231 sec
[COMPARE] MSE: 0.009731649421155453
[COMPARE] Max Abs Diff: 0.4397318363189697
[INFO] Quantized model saved at: ./quantized_model
[SIZE] FP32 Model Size: 0.25 MB
[SIZE] INT8 Model Size (sim): 13.55 MB


In [7]:
import os
import torch
import onnx
import numpy as np

from torchvision.models import mobilenet_v2

# Create model
model = mobilenet_v2(weights='DEFAULT').eval()
dummy_input = torch.randn(1, 3, 224, 224)

onnx_path = "mobilenet_fp32.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    opset_version=13
)

print("✅ FP32 ONNX exported:", onnx_path)

W0429 22:04:52.017000 853 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 13 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/harish/miniconda3/envs/qairt/lib/python3.10/copyreg.py:101: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/home/harish/miniconda3/envs/qairt/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:68: adapt: Assertion `node->hasAttribute(kaxes)` failed: No initializer or constant input to node found


[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ FP32 ONNX exported: mobilenet_fp32.onnx


In [8]:
from aimet_onnx.quantsim import QuantizationSimModel
from aimet_onnx import int8
from aimet_onnx.utils import make_dummy_input

# Load ONNX
onnx_model = onnx.load(onnx_path)

# Create QuantSim
sim = QuantizationSimModel(
    onnx_model,
    param_type=int8,
    activation_type=int8
)

# Calibration (replace with real data for better accuracy)
calib_data = make_dummy_input(onnx_model)
sim.compute_encodings(inputs=[calib_data])

print("✅ AIMET calibration done")

# Export encodings + model
export_dir = "./quantized_model"
os.makedirs(export_dir, exist_ok=True)

sim.export(
    path=export_dir,
    filename_prefix="mobilenet_int8"
)

print("✅ Encodings exported")

2026-04-29 22:05:18,487 - Quant - INFO - Selecting DefaultOpInstanceConfigGenerator to compute the specialized config. hw_version:None
✅ AIMET calibration done
✅ Encodings exported


In [9]:
model = onnx.load(onnx_path)
input_name = model.graph.input[0].name

print("Input name:", input_name)

Input name: input


In [12]:
!pip install onnx==1.14.0 protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 2.0 MB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 10.5 MB/s  0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.1
    Uninstalling protobuf-7.34.1:
      Successfully uninstalled protobuf-7.34.1
  Attempting uninstall: onnx
    Found existing installation: onnx 1.21.0
    Uninstalling onnx-1.21.0:╺━━━━━━━━━━━━━━━━━━━ 1/2 [onnx]
      Successfully uninstalled onnx-1.21.0━━━━━━━━━━━━━━━━━━━ 1/2 [onnx]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]1/2 [onnx]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx-ir 0.2.1 requires onnx>=1.16, but you have onnx 1.14.0 which is incompatible.
onnxscript 0.7.0 requires onnx>=1.17, but you have onnx 1.14.0 which is incompatible.


In [1]:
!source /mnt/d/qcomm/qairt/2.42.0.251225/bin/envsetup.sh && \
qairt-converter \
  --input_network mobilenet_fp32.onnx \
  --quantization_overrides ./quantized_model/mobilenet_int8.encodings \
  --output_path mobilenet_int8.dlc \
  -d input 1,3,224,224

[INFO] QAIRT_SDK_ROOT=/mnt/d/qcomm/qairt/2.42.0.251225
[WARN] QNN_SDK_ROOT/SNPE_ROOT set to QAIRT_SDK_ROOT for backwards compatibility and will be deprecated in a future release.
[INFO] QAIRT SDK environment setup complete
2026-04-29 22:08:46,538 - 278 - INFO - INFO_INITIALIZATION_SUCCESS: 
2026-04-29 22:08:47,266 - 283 - WARNING - Unable to register converter supported Operation [Gelu:Version 20] with your Onnx installation. Converter will bail if Model contains this Op.
2026-04-29 22:08:47,271 - 283 - WARNING - Unable to register converter supported Operation [Inverse:Version 1] with your Onnx installation. Converter will bail if Model contains this Op.
2026-04-29 22:08:47,701 - 283 - WARNING - --desired_input_shape and -d are deprecated. Use --source_model_input_shape or -s for achieving this functionality
2026-04-29 22:08:47,702 - 278 - INFO - Processing user provided quantization encodings: 
2026-04-29 22:08:47,727 - 278 - INFO - Input shape info 
2026-04-29 22:08:47,727 - 283 - W

In [4]:
!ls -lh mobilenet_fp32.onnx*

-rwxrwxrwx 1 harish harish 258K Apr 29 22:04 mobilenet_fp32.onnx
-rwxrwxrwx 1 harish harish  14M Apr 29 22:04 mobilenet_fp32.onnx.data


In [5]:
import os

def get_onnx_total_size(onnx_path):
    """
    Handles both:
    - single ONNX file
    - ONNX + external .data file
    """
    total_size = 0

    # Main ONNX file
    if os.path.exists(onnx_path):
        total_size += os.path.getsize(onnx_path)

    # External data file (common case)
    data_path = onnx_path + ".data"
    if os.path.exists(data_path):
        total_size += os.path.getsize(data_path)

    return total_size


def size_mb(bytes_size):
    return bytes_size / (1024 * 1024)


# Paths
onnx_path = "mobilenet_fp32.onnx"
dlc_path = "mobilenet_int8.dlc"

# Sizes
onnx_size_bytes = get_onnx_total_size(onnx_path)
dlc_size_bytes = os.path.getsize(dlc_path)

# Convert to MB
onnx_size_mb = size_mb(onnx_size_bytes)
dlc_size_mb = size_mb(dlc_size_bytes)

# Compression ratio
compression_ratio = onnx_size_mb / dlc_size_mb

print("===== Model Size Comparison =====")
print(f"FP32 ONNX (total): {onnx_size_mb:.2f} MB")
print(f"INT8 DLC        : {dlc_size_mb:.2f} MB")
print(f"Compression     : {compression_ratio:.2f}x smaller")

===== Model Size Comparison =====
FP32 ONNX (total): 13.57 MB
INT8 DLC        : 4.43 MB
Compression     : 3.06x smaller


In [6]:
import os

def get_total_onnx_size(onnx_path):
    """
    Returns total size of ONNX model:
    - main .onnx file
    - + external .onnx.data if present
    """
    total = 0

    if os.path.exists(onnx_path):
        total += os.path.getsize(onnx_path)

    data_path = onnx_path + ".data"
    if os.path.exists(data_path):
        total += os.path.getsize(data_path)

    return total


def mb(x):
    return x / (1024 * 1024)


# ===== Paths =====
fp32_onnx = "mobilenet_fp32.onnx"
aimet_onnx = "./quantized_model/mobilenet_int8.onnx"
dlc_model = "mobilenet_int8.dlc"

# ===== Sizes =====
fp32_size = get_total_onnx_size(fp32_onnx)
aimet_size = get_total_onnx_size(aimet_onnx)
dlc_size = os.path.getsize(dlc_model)

# ===== Convert =====
fp32_mb = mb(fp32_size)
aimet_mb = mb(aimet_size)
dlc_mb = mb(dlc_size)

# ===== Ratios =====
aimet_vs_fp32 = aimet_mb / fp32_mb
dlc_vs_fp32 = fp32_mb / dlc_mb

# ===== Print =====
print("===== MODEL SIZE COMPARISON =====")
print(f"FP32 ONNX       : {fp32_mb:.2f} MB")
print(f"AIMET ONNX      : {aimet_mb:.2f} MB")
print(f"INT8 DLC        : {dlc_mb:.2f} MB")

print("\n===== RELATIVE =====")
print(f"AIMET vs FP32   : {aimet_vs_fp32:.2f}x larger")
print(f"DLC compression : {dlc_vs_fp32:.2f}x smaller than FP32")

===== MODEL SIZE COMPARISON =====
FP32 ONNX       : 13.57 MB
AIMET ONNX      : 13.55 MB
INT8 DLC        : 4.43 MB

===== RELATIVE =====
AIMET vs FP32   : 1.00x larger
DLC compression : 3.06x smaller than FP32
